# Amazon ML Challenge 2026: Low-RAM Production Pipeline (< 500 MB RAM)
Designed specifically to run seamlessly on default instances () without memory crashes.

In [ ]:
# Cell 1: Low-RAM Environment & Imports
import os
import re
import gc
import time
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import boto3

print("--- Low-RAM Pipeline Initialized ---")


In [ ]:
# Cell 2: Dataset Locator & Text Normalization
DATA_DIR = Path("student_resource/dataset/train")
if not DATA_DIR.exists():
    matches = list(Path(".").glob("**/dataset/train"))
    if matches:
        DATA_DIR = matches[0]

print(f"Dataset Directory: {DATA_DIR.resolve()}")

LEGAL_REGEX = re.compile(r"\b(corp|corporation|inc|incorporated|ltd|limited|pvt|private|co|company|llc|plc|gmbh|sa|bv|srl)\b", flags=re.IGNORECASE)

def clean_text(text: str) -> str:
    if not isinstance(text, str) or pd.isna(text):
        return ""
    text = text.lower()
    text = LEGAL_REGEX.sub("", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_tokens(name: str, address: str):
    full = f"{clean_text(name)} {clean_text(address)}"
    return [t for t in full.split() if len(t) >= 3]

print("Text Normalizer Ready!")


In [ ]:
# Cell 3-UltraCompact: High-Density Inverted Indexer (< 150 MB RAM)
from collections import defaultdict
import gc
import time
import pandas as pd

print("Building Ultra-Compact Inverted Index...")
start_time = time.time()

# Stop-words that occur in millions of addresses/names and add no discriminatory value
STOP_WORDS = {
    "street", "road", "avenue", "drive", "lane", "boulevard", "court", "place",
    "suite", "floor", "building", "unit", "near", "opposite", "behind", "block",
    "north", "south", "east", "west", "central", "main", "state", "city", "town"
}

def extract_informative_tokens(name: str, address: str):
    full = f"{clean_text(name)} {clean_text(address)}"
    tokens = [t for t in full.split() if len(t) >= 4 and t not in STOP_WORDS]
    return tokens

# Compact Inverted Index mapping token -> list of string entity IDs
inverted_index = defaultdict(list)
dtypes = {"entity_id": "string", "business_name": "string", "business_address": "string", "country": "category"}

# Stream Source 2
s2_count = 0
for chunk in pd.read_csv(DATA_DIR / "train_source2.tsv", sep="	", chunksize=100000, dtype=dtypes):
    for eid, bname, baddr in zip(chunk["entity_id"], chunk["business_name"], chunk["business_address"]):
        for token in extract_informative_tokens(bname, baddr):
            inverted_index[token].append(eid)
    s2_count += len(chunk)
    if s2_count % 1000000 == 0:
        print(f"Indexed {s2_count:,} Source 2 records... Vocabulary size: {len(inverted_index):,}")

# Stream Source 3
s3_count = 0
for chunk in pd.read_csv(DATA_DIR / "train_source3.tsv", sep="	", chunksize=100000, dtype=dtypes):
    for eid, bname, baddr in zip(chunk["entity_id"], chunk["business_name"], chunk["business_address"]):
        for token in extract_informative_tokens(bname, baddr):
            inverted_index[token].append(eid)
    s3_count += len(chunk)
    if s3_count % 1000000 == 0:
        print(f"Indexed {s3_count:,} Source 3 records... Vocabulary size: {len(inverted_index):,}")

# Post-process: deduplicate lists to save RAM
print("Deduplicating posting lists...")
for token in list(inverted_index.keys()):
    inverted_index[token] = list(set(inverted_index[token]))

print(f"
SUCCESS! Indexed {s2_count + s3_count:,} records in {time.time() - start_time:.2f} seconds!")
print(f"Total Unique Informative Tokens: {len(inverted_index):,}")
gc.collect()
